# Banking - Serve Credit Scoring Model

Creates a Databricks Model Serving endpoint for the banking credit scoring model.

**Steps:**
1. Promote the latest model version to the "Champion" alias
2. Create (or update) the `banking-credit-scoring` serving endpoint
3. Wait for the endpoint to reach `READY` state
4. Send a test request and display the prediction

In [ ]:
%run ../_setup

In [ ]:
import json
import time

import mlflow
import requests

In [ ]:
# ---------------------------------------------------------------------------
# Promote the latest model version to "Champion"
# ---------------------------------------------------------------------------
INDUSTRY = "banking"
cfg = INDUSTRY_CONFIG[INDUSTRY]
MODEL_NAME = cfg["registered_model_name"]  # banking_credit_scoring_model
ENDPOINT_NAME = "banking-credit-scoring"

client = mlflow.MlflowClient()

# Get the latest version number
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
if not versions:
    raise RuntimeError(f"No versions found for model '{MODEL_NAME}'. Run 02_train_model first.")

latest_version = max(versions, key=lambda v: int(v.version))
print(f"Model        : {MODEL_NAME}")
print(f"Version      : {latest_version.version}")
print(f"Run ID       : {latest_version.run_id}")

# Set the Champion alias
client.set_registered_model_alias(MODEL_NAME, "Champion", latest_version.version)
print(f"Alias set    : Champion -> v{latest_version.version}")

In [ ]:
# ---------------------------------------------------------------------------
# Create or update the serving endpoint
# ---------------------------------------------------------------------------
WORKSPACE_HOST = mlflow.utils.databricks_utils.get_webapp_url()
TOKEN = mlflow.utils.databricks_utils.get_databricks_host_creds().token

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}

ENDPOINT_URL = f"{WORKSPACE_HOST}/api/2.0/serving-endpoints"

endpoint_config = {
    "name": ENDPOINT_NAME,
    "config": {
        "served_entities": [
            {
                "entity_name": MODEL_NAME,
                "entity_version": latest_version.version,
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
            }
        ]
    },
}

# Check whether the endpoint already exists
check_resp = requests.get(f"{ENDPOINT_URL}/{ENDPOINT_NAME}", headers=HEADERS)

if check_resp.status_code == 200:
    # Update the existing endpoint's config
    update_resp = requests.put(
        f"{ENDPOINT_URL}/{ENDPOINT_NAME}/config",
        headers=HEADERS,
        json=endpoint_config["config"],
    )
    update_resp.raise_for_status()
    print(f"Updated existing endpoint: {ENDPOINT_NAME}")
else:
    # Create a new endpoint
    create_resp = requests.post(
        ENDPOINT_URL,
        headers=HEADERS,
        json=endpoint_config,
    )
    create_resp.raise_for_status()
    print(f"Created new endpoint: {ENDPOINT_NAME}")

print(f"Model        : {MODEL_NAME} v{latest_version.version}")
print(f"Workload size: Small")
print(f"Scale to zero: Enabled")

In [ ]:
# ---------------------------------------------------------------------------
# Wait for the endpoint to be ready
# ---------------------------------------------------------------------------
POLL_INTERVAL_SECONDS = 30
TIMEOUT_SECONDS = 1200  # 20 minutes

elapsed = 0
while elapsed < TIMEOUT_SECONDS:
    resp = requests.get(f"{ENDPOINT_URL}/{ENDPOINT_NAME}", headers=HEADERS)
    resp.raise_for_status()
    status = resp.json()

    state = status.get("state", {})
    ready_state = state.get("ready", "UNKNOWN")
    config_update = state.get("config_update", "UNKNOWN")

    print(f"[{elapsed:>4d}s] ready={ready_state}  config_update={config_update}")

    if ready_state == "READY":
        print("\nEndpoint is READY.")
        break

    if ready_state == "FAILED" or config_update == "UPDATE_FAILED":
        raise RuntimeError(
            f"Endpoint entered a failed state: {json.dumps(state, indent=2)}"
        )

    time.sleep(POLL_INTERVAL_SECONDS)
    elapsed += POLL_INTERVAL_SECONDS
else:
    raise TimeoutError(
        f"Endpoint did not become ready within {TIMEOUT_SECONDS} seconds."
    )

In [ ]:
# ---------------------------------------------------------------------------
# Test the endpoint with a sample request
# ---------------------------------------------------------------------------
SCORING_URL = f"{WORKSPACE_HOST}/serving-endpoints/{ENDPOINT_NAME}/invocations"

sample_payload = {
    "dataframe_records": [
        {
            "age": 35,
            "annual_income": 450000,
            "employment_years": 8,
            "loan_amount": 150000,
            "credit_score": 720,
            "num_late_payments": 1,
            "debt_to_income_ratio": 0.33,
            "num_open_accounts": 4,
            "months_since_last_delinquency": 24,
            "education_level": "Bachelors",
            "employment_type": "Permanent",
            "province": "Gauteng",
        }
    ]
}

score_resp = requests.post(SCORING_URL, headers=HEADERS, json=sample_payload)
score_resp.raise_for_status()

result = score_resp.json()
print("Sample request payload:")
print(json.dumps(sample_payload, indent=2))
print("\nEndpoint response:")
print(json.dumps(result, indent=2))

In [ ]:
# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print(f"\n{'='*60}")
print(f"  Banking Credit Scoring - Serving Endpoint")
print(f"{'='*60}")
print(f"  Endpoint name : {ENDPOINT_NAME}")
print(f"  Model         : {MODEL_NAME} v{latest_version.version}")
print(f"  Alias         : Champion")
print(f"  Endpoint URL  : {SCORING_URL}")
print(f"  Workspace     : {WORKSPACE_HOST}")
print(f"{'='*60}")
print(f"\nView the endpoint in the Databricks UI:")
print(f"  {WORKSPACE_HOST}/#/mlflow/endpoints/{ENDPOINT_NAME}")